<a href="https://colab.research.google.com/github/Anushadhirde/Urban-Heat-Island-Change-Detection/blob/main/Split_tiles_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
"""
STEP: Split tiles into train / validation / test sets.

WHY SPATIAL SPLIT, NOT RANDOM SPLIT:
If we split tiles randomly, the SAME physical location in Nagpur (say, a
patch near the airport) could show up as a training tile from 2020 and a
test tile from 2021. Since land cover, terrain, and general character of
a location barely change year to year, the model would basically be
"peeking" at test locations during training — giving you an accuracy
score that looks great but doesn't reflect real performance on genuinely
unseen areas. Splitting by LOCATION instead of by individual tile means:
every tile from a given (row_start, col_start) position, across ALL years
and seasons, goes into the same split (train, val, or test) — so test
locations are truly never seen during training, in any year.

WHAT THIS DOES:
- Reads your tiles_manifest.csv (from the tiling step)
- Finds every unique spatial location (row_start, col_start) — since the
  same location repeats across 18 scenes, there are far fewer unique
  locations than total tiles
- Randomly assigns each unique LOCATION to train/val/test (default 70/15/15)
  using a fixed random seed, so the split is reproducible
- Writes a new manifest CSV with a "split" column added, so you know
  exactly which set every tile belongs to
- Does NOT move or copy the actual .npy files — your training code will
  just read this manifest to know which tile_ids belong to which split

BEFORE YOU RUN:
- Confirm MANIFEST_PATH matches where your tiling script saved it.
"""

import csv
import random

MANIFEST_PATH = "/content/drive/MyDrive/DATASET/TILES/tiles_manifest.csv"
OUTPUT_PATH = "/content/drive/MyDrive/DATASET/TILES/tiles_manifest_split.csv"

TRAIN_FRACTION = 0.70
VAL_FRACTION = 0.15
# TEST_FRACTION is whatever's left: 1 - TRAIN_FRACTION - VAL_FRACTION = 0.15

RANDOM_SEED = 42  # fixed seed so the split is the same every time you run this


def main():
    with open(MANIFEST_PATH, "r") as f:
        reader = csv.DictReader(f)
        rows = list(reader)

    if not rows:
        print(f"No rows found in {MANIFEST_PATH} — check the path.")
        return

    print(f"Loaded {len(rows)} tiles from manifest.")

    # find every unique spatial location
    locations = sorted(set((row["row_start"], row["col_start"]) for row in rows))
    print(f"Found {len(locations)} unique spatial locations "
          f"(each repeats across up to 18 year/season scenes).")

    # shuffle locations (not tiles!) with a fixed seed, then split
    rng = random.Random(RANDOM_SEED)
    shuffled_locations = locations[:]
    rng.shuffle(shuffled_locations)

    n = len(shuffled_locations)
    n_train = int(n * TRAIN_FRACTION)
    n_val = int(n * VAL_FRACTION)

    train_locations = set(shuffled_locations[:n_train])
    val_locations = set(shuffled_locations[n_train:n_train + n_val])
    test_locations = set(shuffled_locations[n_train + n_val:])

    print(f"\nLocation split: {len(train_locations)} train, "
          f"{len(val_locations)} val, {len(test_locations)} test")

    # assign every tile a split based on its location
    counts = {"train": 0, "val": 0, "test": 0}
    for row in rows:
        loc = (row["row_start"], row["col_start"])
        if loc in train_locations:
            row["split"] = "train"
        elif loc in val_locations:
            row["split"] = "val"
        else:
            row["split"] = "test"
        counts[row["split"]] += 1

    fieldnames = list(rows[0].keys())
    with open(OUTPUT_PATH, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

    print(f"\nTile split (should roughly match the {int(TRAIN_FRACTION*100)}/"
          f"{int(VAL_FRACTION*100)}/{int((1-TRAIN_FRACTION-VAL_FRACTION)*100)} target):")
    total = len(rows)
    for split_name, count in counts.items():
        print(f"  {split_name}: {count} tiles ({100*count/total:.1f}%)")

    print(f"\nDone. Saved split manifest to {OUTPUT_PATH}")


if __name__ == "__main__":
    main()

Loaded 3942 tiles from manifest.
Found 219 unique spatial locations (each repeats across up to 18 year/season scenes).

Location split: 153 train, 32 val, 34 test

Tile split (should roughly match the 70/15/15 target):
  train: 2754 tiles (69.9%)
  val: 576 tiles (14.6%)
  test: 612 tiles (15.5%)

Done. Saved split manifest to /content/drive/MyDrive/DATASET/TILES/tiles_manifest_split.csv
